In [36]:
from pyspark.sql import functions as F

CUENTA = "stolistfabri01"
BRONZE = f"abfss://bronze@{CUENTA}.dfs.core.windows.net/olist"
SILVER = f"abfss://silver@{CUENTA}.dfs.core.windows.net/olist"

# R1: declaramos los tipos. Los códigos postales como STRING para no perder ceros a la izquierda.
esquema_sellers = "seller_id STRING, seller_zip_code_prefix STRING, seller_city STRING, seller_state STRING"
esquema_customers = ("customer_id STRING, customer_unique_id STRING, customer_zip_code_prefix STRING, "
                     "customer_city STRING, customer_state STRING")

sellers   = spark.read.csv(f"{BRONZE}/olist_sellers_dataset.csv",   header=True, schema=esquema_sellers)
customers = spark.read.csv(f"{BRONZE}/olist_customers_dataset.csv", header=True, schema=esquema_customers)

print("sellers:", sellers.count(), "| customers:", customers.count())
display(sellers.limit(5))

StatementMeta(sparkolist, 14, 30, Finished, Available, Finished, False)

sellers: 3095 | customers: 99441


SynapseWidget(Synapse.DataFrame, ca5c1b76-6401-44d0-b5b9-51c0d321e5ff)

In [37]:
import unicodedata
from pyspark.sql.types import StringType

def quitar_tildes(texto):
    if texto is None:
        return None
    descompuesto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))

quitar_tildes_udf = F.udf(quitar_tildes, StringType())

def normalizar(col):
    c = F.lower(F.trim(col))
    c = F.regexp_replace(c, r"\s+", " ")
    c = F.regexp_replace(c, "[´`’‘]", "'")
    return quitar_tildes_udf(c)

pares = (
    sellers.select(F.col("seller_city").alias("ciudad_origen"),
                   F.col("seller_state").alias("estado"),
                   F.col("seller_zip_code_prefix").alias("cp"))
    .unionByName(
        customers.select(F.col("customer_city").alias("ciudad_origen"),
                         F.col("customer_state").alias("estado"),
                         F.col("customer_zip_code_prefix").alias("cp"))
    )
    .withColumn("estado", F.upper(F.trim("estado")))
    .distinct()
    .withColumn("ciudad_norm", normalizar(F.col("ciudad_origen")))
)

print("combinaciones distintas (ciudad, estado, cp):", pares.count())
display(pares.filter(F.col("ciudad_origen") != F.col("ciudad_norm")).limit(20))

StatementMeta(sparkolist, 14, 31, Finished, Available, Finished, False)

combinaciones distintas (ciudad, estado, cp): 15249


SynapseWidget(Synapse.DataFrame, 5e39f803-8cf1-4ff7-8173-cd26ffb2d2e9)

In [38]:
pares_r4 = (
    pares
    # Regla A: todo lo que va después de "/", "\" o "," se corta
    .withColumn("ciudad_r4", F.regexp_replace("ciudad_norm", r"\s*[/\\,].*$", ""))
    # Regla B: si termina en [espacio, guion o coma] + la sigla de SU PROPIO estado, se corta
    .withColumn("ciudad_r4", F.expr("""
        CASE WHEN ciudad_r4 RLIKE concat('[ ,-]', lower(estado), '$')
             THEN trim(TRAILING ' ,-' FROM substring(ciudad_r4, 1, length(ciudad_r4) - 2))
             ELSE ciudad_r4 END
    """))
)

cambios_r4 = pares_r4.filter(F.col("ciudad_r4") != F.col("ciudad_norm"))
print("Ciudades corregidas por R4:", cambios_r4.count())
display(cambios_r4.select("ciudad_origen", "estado", "ciudad_norm", "ciudad_r4").orderBy("ciudad_r4"))

StatementMeta(sparkolist, 14, 32, Finished, Available, Finished, False)

Ciudades corregidas por R4: 26


SynapseWidget(Synapse.DataFrame, f6ddfabd-b07c-4726-8646-640046257634)

In [39]:
UFS = ["ac","al","ap","am","ba","ce","df","es","go","ma","mt","ms","mg","pa","pb",
       "pr","pe","pi","rj","rn","rs","ro","rr","sc","sp","se","to"]

# A) Después de R4, ¿alguna sigue terminando en una sigla que NO es la de su estado?
patron_uf = r"[ ,-](" + "|".join(UFS) + r")$"
sospechosas = pares_r4.filter(F.col("ciudad_r4").rlike(patron_uf))
print("A) Terminan en una sigla que NO es la de su estado:", sospechosas.count())
display(sospechosas.select("ciudad_origen", "estado", "ciudad_r4"))

# B) Toda ciudad cuya última palabra tiene 2 letras (errores + posibles ciudades reales)
dos_letras = (pares_r4.filter(F.col("ciudad_norm").rlike(r" [a-z]{2}$"))
              .select("ciudad_norm", "estado", "ciudad_r4").distinct().orderBy("ciudad_norm"))
print("B) Última palabra de 2 letras:", dos_letras.count())
display(dos_letras)

StatementMeta(sparkolist, 14, 33, Finished, Available, Finished, False)

A) Terminan en una sigla que NO es la de su estado: 2


SynapseWidget(Synapse.DataFrame, f5d7e616-e70a-4263-8c38-abdece27ec8f)

B) Última palabra de 2 letras: 18


SynapseWidget(Synapse.DataFrame, a394cd5d-4ce6-46b5-90c7-3949ddeb19fb)

In [40]:
# A) Caracteres fuera de lo esperado. Después de normalizar, una ciudad solo debería
#    tener letras a-z, espacios, apóstrofos y guiones.
raros = pares_r4.filter(~F.col("ciudad_r4").rlike(r"^[a-z' -]+$"))
print("A) Con caracteres fuera de [a-z ' -]:", raros.count())
display(raros.select("ciudad_origen", "estado", "ciudad_r4").orderBy("ciudad_r4"))

# B) Nombres demasiado cortos (posibles siglas o abreviaturas en vez del nombre)
cortas = pares_r4.filter(F.length("ciudad_r4") <= 3)
print("B) Nombres de 3 caracteres o menos:", cortas.count())
display(cortas.select("ciudad_origen", "estado", "ciudad_r4").orderBy("ciudad_r4"))

# C) Ciudades con guion (muchas son reales: hay que mirarlas, no corregirlas a ciegas)
guiones = pares_r4.filter(F.col("ciudad_r4").contains("-")).select("ciudad_r4", "estado").distinct()
print("C) Con guion:", guiones.count())
display(guiones.orderBy("ciudad_r4"))

# D) Ciudad vacía o nula
nulas = pares_r4.filter(F.col("ciudad_r4").isNull() | (F.col("ciudad_r4") == ""))
print("D) Vacías o nulas:", nulas.count())
display(nulas)

StatementMeta(sparkolist, 14, 34, Finished, Available, Finished, False)

A) Con caracteres fuera de [a-z ' -]: 4


SynapseWidget(Synapse.DataFrame, 79c0e1e3-b769-49d5-955a-e2c44e0fc5f6)

B) Nombres de 3 caracteres o menos: 69


SynapseWidget(Synapse.DataFrame, 8fa01ff4-095d-47a1-a881-39d04fa90ff0)

C) Con guion: 21


SynapseWidget(Synapse.DataFrame, ded0f142-2eec-42a5-aee6-6958499de62d)

D) Vacías o nulas: 0


SynapseWidget(Synapse.DataFrame, 9c3b793c-5975-42c3-b681-36dfaa3dadf0)

In [41]:
sospechosos = ["sbc/sp", "sp / sp", "sp", "aguas claras df",
               "04482255", "vendas@creditparts.com.br", "arraial d'ajuda (porto seguro)"]

# 1) ¿De dónde vienen (vendedor o cliente) y qué código postal tienen?
vend = (sellers.withColumn("c", normalizar(F.col("seller_city")))
        .filter(F.col("c").isin(sospechosos))
        .select(F.lit("vendedor").alias("tabla"), F.col("seller_city").alias("ciudad"),
                F.col("seller_state").alias("estado"), F.col("seller_zip_code_prefix").alias("cp")))
cli = (customers.withColumn("c", normalizar(F.col("customer_city")))
       .filter(F.col("c").isin(sospechosos))
       .select(F.lit("cliente").alias("tabla"), F.col("customer_city").alias("ciudad"),
               F.col("customer_state").alias("estado"), F.col("customer_zip_code_prefix").alias("cp")))
casos = vend.unionByName(cli)
display(casos.orderBy("ciudad"))

# 2) Evidencia: ¿qué ciudad tienen los CLIENTES con esos mismos códigos postales?
cps = [fila.cp for fila in casos.select("cp").distinct().collect()]
evidencia = (customers.filter(F.col("customer_zip_code_prefix").isin(cps))
             .groupBy("customer_zip_code_prefix", "customer_city", "customer_state").count()
             .orderBy("customer_zip_code_prefix", F.desc("count")))
display(evidencia)

StatementMeta(sparkolist, 14, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72525e41-bb73-41ee-927c-19f5fd1a8983)

SynapseWidget(Synapse.DataFrame, 656dad23-a92f-45a3-b045-e678ccaa3c29)

In [46]:
REFERENCIA = f"abfss://bronze@{CUENTA}.dfs.core.windows.net/referencia"

correcciones = spark.read.csv(f"{REFERENCIA}/correccion_ciudades.csv", header=True,
    schema=("ciudad_r4 STRING, estado STRING, cp STRING, ciudad_corregida STRING, estado_corregido STRING, "
            "municipio_corregido STRING, motivo STRING, evidencia STRING, fecha_correccion DATE"))

ibge = (spark.read.csv(f"{REFERENCIA}/ibge_distritos.csv", header=True,
        schema="uf STRING, cod_municipio_ibge INT, municipio STRING, cod_distrito_ibge INT, distrito STRING")
        .withColumn("municipio_norm", normalizar(F.col("municipio")))
        .withColumn("distrito_norm",  normalizar(F.col("distrito"))))

pares_c = (
    pares_r4.join(correcciones.select("ciudad_r4", "estado", "cp", "ciudad_corregida",
                                      "estado_corregido", "municipio_corregido"),
                  on=["ciudad_r4", "estado", "cp"], how="left")
    .withColumn("localidad",        F.coalesce("ciudad_corregida", "ciudad_r4"))
    .withColumn("estado_std",       F.coalesce("estado_corregido", "estado"))
    .withColumn("municipio_manual", F.col("municipio_corregido"))
    .withColumn("es_manual",        F.col("ciudad_corregida").isNotNull())
)
print("Corregidas a mano (esperado 59):", pares_c.filter("es_manual").count())
print("  con municipio asignado a mano (esperado 49):", pares_c.filter(F.col("municipio_manual").isNotNull()).count())

StatementMeta(sparkolist, 14, 46, Finished, Available, Finished, False)

Corregidas a mano (esperado 59): 98
  con municipio asignado a mano (esperado 49): 88


In [47]:
from pyspark.sql import Window

def clave(col):
    return F.regexp_replace(col, r"[ '\-]", "")

nombres = (
    ibge.select(F.col("uf").alias("estado_std"), F.col("municipio_norm").alias("nombre"),
                "municipio_norm", "cod_municipio_ibge", F.lit(1).alias("es_mun"))
    .unionByName(
        ibge.select(F.col("uf").alias("estado_std"), F.col("distrito_norm").alias("nombre"),
                    "municipio_norm", "cod_municipio_ibge", F.lit(0).alias("es_mun")))
    .distinct()
)

ref = (
    nombres.groupBy("estado_std", "nombre")
    .agg(F.max("es_mun").alias("es_municipio"),
         F.countDistinct("cod_municipio_ibge").alias("n_municipios"),
         F.first(F.when(F.col("es_mun") == 1, F.col("municipio_norm")), ignorenulls=True).alias("mun_propio"),
         F.first(F.when(F.col("es_mun") == 1, F.col("cod_municipio_ibge")), ignorenulls=True).alias("cod_propio"),
         F.first("municipio_norm").alias("mun_unico"),
         F.first("cod_municipio_ibge").alias("cod_unico"))
    .withColumn("tipo_ref", F.when(F.col("es_municipio") == 1, "municipio")
                             .when(F.col("n_municipios") == 1, "distrito")
                             .otherwise("distrito_ambiguo"))
    .withColumn("municipio", F.when(F.col("tipo_ref") == "municipio", F.col("mun_propio"))
                              .when(F.col("tipo_ref") == "distrito", F.col("mun_unico")))
    .withColumn("cod_municipio_ibge", F.when(F.col("tipo_ref") == "municipio", F.col("cod_propio"))
                                       .when(F.col("tipo_ref") == "distrito", F.col("cod_unico")))
    .withColumn("clave", clave(F.col("nombre")))
    .select("estado_std", "nombre", "clave", "tipo_ref", "municipio", "cod_municipio_ibge")
)
ref_ok = ref.filter(F.col("tipo_ref").isin("municipio", "distrito"))
brasilia = ref.filter("estado_std = 'DF' AND nombre = 'brasilia'").select("municipio", "cod_municipio_ibge").first()

loc = (pares_c.select("localidad", "estado_std").distinct()
       .withColumn("clave", clave(F.col("localidad"))))

exacta = ref.select("estado_std", F.col("nombre").alias("localidad"), "tipo_ref",
                    F.col("municipio").alias("mun_exacta"), F.col("cod_municipio_ibge").alias("cod_exacta"))

punt = (ref_ok.groupBy("estado_std", "clave")
        .agg(F.countDistinct("municipio").alias("n"),
             F.first("municipio").alias("mun_punt"), F.first("cod_municipio_ibge").alias("cod_punt"),
             F.first("nombre").alias("nombre_punt"))
        .filter("n = 1").drop("n"))

paso = (loc.join(exacta, ["localidad", "estado_std"], "left")
           .join(punt, ["estado_std", "clave"], "left"))

pendientes = paso.filter(F.col("tipo_ref").isNull() & F.col("mun_punt").isNull() & (F.col("estado_std") != "DF"))
cand = (pendientes.select("localidad", "estado_std")
        .join(ref_ok.select("estado_std", F.col("nombre").alias("nombre_fuzzy"),
                            "municipio", "cod_municipio_ibge"), "estado_std")
        .withColumn("distancia", F.levenshtein("localidad", "nombre_fuzzy"))
        # AJUSTE (a): distancia 1 para cualquier nombre de 6+ letras; distancia 2 solo para nombres de 10+ letras
        .filter((F.length("localidad") >= 6) &
                ((F.col("distancia") <= 1) |
                 ((F.col("distancia") <= 2) & (F.length("localidad") >= 10)))))
w = Window.partitionBy("localidad", "estado_std")
fuzzy = (cand.withColumn("d_min", F.min("distancia").over(w))
         .filter(F.col("distancia") == F.col("d_min"))
         .groupBy("localidad", "estado_std")
         .agg(F.countDistinct("municipio").alias("n"), F.first("nombre_fuzzy").alias("nombre_fuzzy"),
              F.first("municipio").alias("mun_fuzzy"), F.first("cod_municipio_ibge").alias("cod_fuzzy"),
              F.first("distancia").alias("distancia"))
         .filter("n = 1").drop("n"))

resueltas = (
    paso.join(fuzzy, ["localidad", "estado_std"], "left")
    .withColumn("metodo",
        F.when(F.col("tipo_ref").isin("municipio", "distrito"), F.col("tipo_ref"))
         .when(F.col("estado_std") == "DF", "region_df")
         .when(F.col("mun_punt").isNotNull(), "puntuacion")
         .when(F.col("mun_fuzzy").isNotNull(), "fuzzy")
         .when(F.col("tipo_ref") == "distrito_ambiguo", "distrito_ambiguo")
         .otherwise("sin_resolver"))
    .withColumn("municipio",
        F.when(F.col("metodo").isin("municipio", "distrito"), F.col("mun_exacta"))
         .when(F.col("metodo") == "region_df", F.lit(brasilia["municipio"]))
         .when(F.col("metodo") == "puntuacion", F.col("mun_punt"))
         .when(F.col("metodo") == "fuzzy", F.col("mun_fuzzy")))
    .withColumn("cod_municipio_ibge",
        F.when(F.col("metodo").isin("municipio", "distrito"), F.col("cod_exacta"))
         .when(F.col("metodo") == "region_df", F.lit(brasilia["cod_municipio_ibge"]))
         .when(F.col("metodo") == "puntuacion", F.col("cod_punt"))
         .when(F.col("metodo") == "fuzzy", F.col("cod_fuzzy")))
    .withColumn("nombre_ibge", F.coalesce("nombre_punt", "nombre_fuzzy"))
    .select("localidad", "estado_std", "municipio", "cod_municipio_ibge", "metodo", "nombre_ibge", "distancia")
)

display(resueltas.groupBy("metodo").count().orderBy(F.desc("count")))

StatementMeta(sparkolist, 14, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3e7d0c9-ec57-4830-b05a-d02baca10082)

In [48]:
# AJUSTE (b): vecinos confiables = clientes resueltos SIN fuzzy
confiables = resueltas.filter(F.col("metodo").isin("municipio", "distrito", "region_df", "puntuacion"))

cli = customers.select(F.col("customer_city").alias("ciudad_origen"),
                       F.upper(F.trim("customer_state")).alias("estado"),
                       F.col("customer_zip_code_prefix").alias("cp"))
cli_res = (cli.join(pares_c.select("ciudad_origen", "estado", "cp", "localidad", "estado_std"),
                    ["ciudad_origen", "estado", "cp"])
              .join(confiables.select("localidad", "estado_std", "municipio"), ["localidad", "estado_std"])
              .withColumn("cp4", F.substring("cp", 1, 4)))

def evidencia(col_cp, sufijo):
    w = Window.partitionBy(col_cp)
    return (cli_res.groupBy(col_cp, "estado_std", "municipio").count()
            .withColumn("total", F.sum("count").over(w))
            .withColumn("apoyo", F.round(F.col("count") / F.col("total"), 2))
            .withColumn("rk", F.row_number().over(w.orderBy(F.desc("count"))))
            .filter("rk = 1")
            .select(col_cp,
                    F.col("municipio").alias("mun" + sufijo), F.col("estado_std").alias("est" + sufijo),
                    F.col("count").alias("a_favor" + sufijo), F.col("total").alias("total" + sufijo),
                    F.col("apoyo").alias("apoyo" + sufijo)))

e5 = evidencia("cp", "_5")     # vecinos con el mismo CP de 5 dígitos
e4 = evidencia("cp4", "_4")    # AJUSTE (c): vecinos de la zona de 4 dígitos

# --- 1) Propuestas para lo que sigue sin resolver (y sin corrección manual) ---
pend = (pares_c.filter(F.col("municipio_manual").isNull())
        .join(resueltas.filter(F.col("metodo").isin("sin_resolver", "distrito_ambiguo"))
                       .select("localidad", "estado_std", "metodo"), ["localidad", "estado_std"])
        .select("ciudad_r4", "estado", "cp", "localidad", "estado_std", "metodo").distinct()
        .withColumn("cp4", F.substring("cp", 1, 4)))

propuestas = (
    pend.join(e5, "cp", "left").join(e4, "cp4", "left")
    .withColumn("nivel", F.when(F.col("mun_5").isNotNull(), "cp5").when(F.col("mun_4").isNotNull(), "cp4"))
    .withColumn("municipio_sugerido", F.coalesce("mun_5", "mun_4"))
    .withColumn("estado_sugerido",    F.coalesce("est_5", "est_4"))
    .withColumn("a_favor",            F.coalesce("a_favor_5", "a_favor_4"))
    .withColumn("total",              F.coalesce("total_5", "total_4"))
    .withColumn("apoyo",              F.coalesce("apoyo_5", "apoyo_4"))
    .withColumn("calidad",
        F.when((F.col("nivel") == "cp5") & (F.col("apoyo") >= 0.8) & (F.col("total") >= 3), "fuerte")
         .when((F.col("nivel") == "cp4") & (F.col("apoyo") >= 0.8) & (F.col("total") >= 3), "fuerte_cp4")
         .when(F.col("nivel").isNotNull(), "dudosa")
         .otherwise("sin_evidencia"))
)
display(propuestas.groupBy("calidad").count())
display(propuestas.select("ciudad_r4", "estado", "cp", "metodo", "nivel", "municipio_sugerido",
                          "estado_sugerido", "a_favor", "total", "apoyo", "calidad")
                  .orderBy("calidad", "estado", "ciudad_r4"))

# --- 2) Control del fuzzy: ¿los vecinos del CP están de acuerdo? ---
fuzzy_chk = (pares_c.filter(F.col("municipio_manual").isNull())
    .join(resueltas.filter("metodo = 'fuzzy'")
                   .select("localidad", "estado_std", "municipio", "nombre_ibge", "distancia"),
          ["localidad", "estado_std"])
    .join(e5, "cp", "left")
    .withColumn("coincide", F.col("municipio") == F.col("mun_5"))
    .select("ciudad_r4", "estado", "cp", "nombre_ibge", "municipio", "distancia",
            "mun_5", "est_5", "a_favor_5", "total_5", "coincide"))
display(fuzzy_chk.filter(~F.coalesce(F.col("coincide"), F.lit(False))).orderBy("estado", "ciudad_r4"))

StatementMeta(sparkolist, 14, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d65da381-913e-4f2c-b876-6dec0f07d4ad)

SynapseWidget(Synapse.DataFrame, f91d3363-3377-4226-bbc3-999a0013a8ea)

SynapseWidget(Synapse.DataFrame, 52d08042-2847-477a-8f56-73cb896ee79f)

In [49]:
# Código IBGE de cada municipio (para los municipios asignados a mano)
cod_mun = (ibge.select(F.col("uf").alias("estado_std"),
                       F.col("municipio_norm").alias("municipio_manual"),
                       F.col("cod_municipio_ibge").alias("cod_manual"))
           .distinct())

mapa = (
    pares_c
    .join(resueltas.select("localidad", "estado_std",
                           F.col("municipio").alias("mun_auto"),
                           F.col("cod_municipio_ibge").alias("cod_auto"),
                           F.col("metodo").alias("metodo_auto"), "distancia"),
          ["localidad", "estado_std"], "left")
    .join(cod_mun, ["estado_std", "municipio_manual"], "left")
    # Prioridad: municipio manual > resolución automática (exacto, distrito, DF, puntuación, fuzzy)
    .withColumn("municipio",          F.coalesce("municipio_manual", "mun_auto"))
    .withColumn("cod_municipio_ibge", F.coalesce("cod_manual", "cod_auto"))
    .withColumn("metodo",
        F.when(F.col("municipio_manual").isNotNull(), "manual")
         .when(F.col("es_manual"), F.concat(F.lit("manual+"), F.col("metodo_auto")))
         .otherwise(F.col("metodo_auto")))
    .select("ciudad_origen",
            F.col("estado").alias("estado_origen"),
            "cp",
            "localidad",
            F.col("estado_std").alias("estado"),
            "municipio",
            "cod_municipio_ibge",
            "metodo",
            "distancia")
    .withColumn("fecha_carga", F.current_timestamp())
)

# --- Controles antes de escribir ---
print("Filas del mapa:", mapa.count(), "| combinaciones de origen:", pares.count(), "(deben ser iguales)")
print("Llaves duplicadas (debe ser 0):",
      mapa.groupBy("ciudad_origen", "estado_origen", "cp").count().filter("count > 1").count())
print("Municipio manual sin código IBGE (debe ser 0):",
      mapa.filter((F.col("metodo") == "manual") & F.col("cod_municipio_ibge").isNull()).count())
display(mapa.groupBy("metodo").agg(F.count("*").alias("combinaciones"),
                                   F.sum(F.col("municipio").isNull().cast("int")).alias("sin_municipio"))
            .orderBy(F.desc("combinaciones")))

# --- Escribir en plata ---
mapa.coalesce(1).write.mode("overwrite").parquet(f"{SILVER}/mapa_ciudades")
print("Escrito en:", f"{SILVER}/mapa_ciudades")

StatementMeta(sparkolist, 14, 49, Finished, Available, Finished, False)

Filas del mapa: 15249 | combinaciones de origen: 15249 (deben ser iguales)
Llaves duplicadas (debe ser 0): 0
Municipio manual sin código IBGE (debe ser 0): 0


SynapseWidget(Synapse.DataFrame, ad0752f0-cb38-40bd-9f79-d39c8d91ff66)

Escrito en: abfss://silver@stolistfabri01.dfs.core.windows.net/olist/mapa_ciudades
